In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import spatialdata as sd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Optional, Tuple, Literal
import math

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
tissue_colors = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

colors_dutch = ['#F79F1F',
 '#1289A7',
 '#009432',
 '#9980FA',
 '#EA2027',
 '#833471',
 '#1B1464']

colors_dutch_y = ['#F79F1F',
 '#1289A7',
 '#A3CB38',
 '#9980FA',
 '#ED4C67',
 '#833471',
 '#D980FA',
 '#009432',
 '#1B1464',
 '#0652DD',
'#EA2027',
 '#EE5A24']

colors_dutch_long = [
    '#FFC312', '#C4E538', '#12CBC4', '#FDA7DF', '#ED4C67',
    '#F79F1F', '#A3CB38', '#1289A7', '#D980FA', '#B53471',
    '#EE5A24', '#009432', '#0652DD', '#9980FA', '#833471',
    '#EA2027', '#006266', '#1B1464', '#5758BB', '#6F1E51'
]

# Mann-Whitney and scCODA (overlap composition)

In [ ]:

ADATA_PATH  = str(P.processed.adata.all_cells / P.fn.all_cells_morans)
SDATA_DIR   = str(P.interim.sdata)
all_cells_adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded {all_cells_adata.n_obs} cells")

In [ ]:
output_dir = str(P.results.figures / "figure_3")
os.makedirs(output_dir, exist_ok=True)


In [ ]:
DENSE_FEATURES = [
    "GDF15_dense",
    "stemness_score_dense",
    "senepy_intestine_epi_0_dense",
]

FEATURE_LABELS = {
    "GDF15_dense":                  "GDF15",
    "stemness_score_dense":         "Stemness",
    "senepy_intestine_epi_0_dense": "SenePy",
}

FEATURE_COLORS = {
    "GDF15_dense":                  "#d62728",
    "stemness_score_dense":         "#1f78b4",
    "senepy_intestine_epi_0_dense": "#f0c929",
}

OVERLAP_PAIRS = [
    ("GDF15_dense", "stemness_score_dense",                   "#7b2d8e"),
    ("stemness_score_dense", "senepy_intestine_epi_0_dense",  "#2ca02c"),
]

TRIPLE_COLOR    = "#000000"
COLOR_OTHER_EPI = "#e0e0e0"
COLOR_NON_EPI   = "#f5f5f5"
ALPHA_DENSE     = 0.9
ALPHA_OTHER_EPI = 0.4
ALPHA_NON_EPI   = 0.15
ALPHA_BOTH      = 1.0
ALPHA_ONE_OF    = 0.7
DOT_SIZE_DENSE  = 8
DOT_SIZE_OTHER  = 2
DOT_SIZE_NON_EPI = 0.5
DOT_SIZE_BOTH   = 12
DOT_SIZE_ONE_OF = 6
DOT_SIZE_TRIPLE = 14
FIGSIZE_PER_SUBPLOT = (4.5, 4.5)

TISSUE_ORDER = ['Dist_N', 'Adj_N', 'AD', 'CA']

# Morans overlap categories

CATEGORY_ORDER = [
    "GDF15 only",
    "Stemness only",
    "SenePy only",
    "GDF15 ∩ Stemness",
    "Stemness ∩ SenePy",
    "GDF15 ∩ SenePy",
    "All three",
]


CATEGORY_COLORS = {
    "GDF15 only":         "#EA2027",
    "Stemness only":      "#44c3e3",
    "SenePy only":        "#F79F1F",
    "GDF15 ∩ Stemness":   "#5758BB",
    "Stemness ∩ SenePy":  "#046e27",
    "GDF15 ∩ SenePy":     "#EE5A24",  
    "All three":          "#000000",
}

In [ ]:
def _classify_overlap(row):
    """Classify an epithelial cell into morans overlap categories."""
    g = row["_g"]
    s = row["_s"]
    p = row["_p"]
    if g and s and p:
        return "All three"
    elif g and s:
        return "GDF15 ∩ Stemness"
    elif g and p:
        return "GDF15 ∩ SenePy"
    elif s and p:
        return "Stemness ∩ SenePy"
    elif g:
        return "GDF15 only"
    elif s:
        return "Stemness only"
    elif p:
        return "SenePy only"
    else:
        return "None"

import matplotlib.patches as mpatches

In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
import matplotlib.patches as mpatches

# Classify cells
obs = all_cells_adata.obs.copy()
epi = obs[obs['annotation_final_fine_cd8'] == 'Epithelial'].copy()

feat_a, feat_b, feat_c = DENSE_FEATURES
epi['_g'] = epi[feat_a] == True
epi['_s'] = epi[feat_b] == True
epi['_p'] = epi[feat_c] == True
epi['_category'] = epi.apply(_classify_overlap, axis=1)
epi = epi[epi['_category'] != 'None'].copy()

# Core-level counts and proportions
patient_counts_overlap = epi.groupby(
    ['patient_id', 'tissue_type_cell_level_normal_split']
)['_category'].value_counts().unstack(fill_value=0)
patient_counts_overlap = patient_counts_overlap[patient_counts_overlap.sum(axis=1) > 0]

for cat in CATEGORY_ORDER:
    if cat not in patient_counts_overlap.columns:
        patient_counts_overlap[cat] = 0
patient_counts_overlap = patient_counts_overlap[CATEGORY_ORDER]

patient_totals = patient_counts_overlap.sum(axis=1)
patient_props_overlap = patient_counts_overlap.div(patient_totals, axis=0)
patient_props_overlap['tissue_type'] = patient_counts_overlap.index.get_level_values(1)

tissue_types = ['Dist_N', 'Adj_N', 'AD', 'CA']

cat_results = []
for t1, t2 in combinations(tissue_types, 2):
    for cat in CATEGORY_ORDER:
        group1 = patient_props_overlap[patient_props_overlap['tissue_type'] == t1][cat]
        group2 = patient_props_overlap[patient_props_overlap['tissue_type'] == t2][cat]
        if len(group1) > 0 and len(group2) > 0:
            stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
            cat_results.append({
                'tissue_1': t1, 'tissue_2': t2,
                'category': cat, 'U': stat, 'p': p
            })

cat_df = pd.DataFrame(cat_results)
cat_df['p_adj'] = multipletests(cat_df['p'], method='fdr_bh')[1]
print(cat_df.to_string())

# Heatmap
sig_matrix = cat_df.pivot_table(
    index='category', columns=['tissue_1', 'tissue_2'], values='p_adj'
)

def sig_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

sig_labels = sig_matrix.applymap(sig_label)

sig_matrix.columns = [f'{t1} vs {t2}' for t1, t2 in sig_matrix.columns]
sig_labels.columns = sig_matrix.columns

sig_matrix = sig_matrix.loc[[c for c in CATEGORY_ORDER if c in sig_matrix.index]]
sig_labels = sig_labels.loc[sig_matrix.index]

col_order = [
    'Dist_N vs Adj_N', 'Dist_N vs AD', 'Dist_N vs CA',
    'Adj_N vs AD', 'Adj_N vs CA', 'AD vs CA'
]
sig_matrix = sig_matrix[col_order]
sig_labels = sig_labels[col_order]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    -np.log10(sig_matrix.astype(float)),
    annot=sig_labels, fmt='', cmap='Reds',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '-log10(p_adj)'},
    ax=ax, yticklabels=False
)
ax.tick_params(axis='y', left=False)
ax.set_ylabel('')

fig.canvas.draw()
bbox = ax.get_window_extent()
ncols = len(col_order)
nrows = len(sig_matrix)
data_units_per_pixel_x = ncols / bbox.width
row_height_px = bbox.height / nrows
square_width = row_height_px * data_units_per_pixel_x

for i, cat in enumerate(sig_matrix.index):
    color = CATEGORY_COLORS[cat]
    rect = mpatches.Rectangle(
        (-square_width, i), square_width, 1,
        facecolor=color, edgecolor='white', linewidth=0.5,
        clip_on=False
    )
    ax.add_patch(rect)
    ax.text(-square_width - 0.15, i + 0.5, cat, ha='right', va='center',
            fontsize=12, fontweight='bold', clip_on=False)

ax.set_xlabel('Comparison', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "overlap_composition_mannwhitney_heatmap_patient.pdf"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import anndata as ad
import sccoda.util.comp_ana as mod

# Build pt-level AnnData
patient_adata_overlap = ad.AnnData(
    X=patient_counts_overlap.values,
    obs=patient_counts_overlap.index.to_frame(index=False).rename(
        columns={patient_counts_overlap.index.names[0]: 'patient_id',
                 patient_counts_overlap.index.names[1]: 'tissue_type'}
    ),
)
patient_adata_overlap.var_names = patient_counts_overlap.columns.astype(str)
patient_adata_overlap.obs.index = patient_adata_overlap.obs.index.astype(str)

# Use largest category as reference
ref_cell_type = patient_counts_overlap.sum().idxmax()
print(f"Reference cell type: {ref_cell_type}")


pair_source = {
    'Dist_N': ['Adj_N', 'AD', 'CA'],   
    'Adj_N':          ['AD', 'CA'],          
    'AD':         ['CA'],                
}

sccoda_results = {}
for ref_tissue in ['Dist_N', 'Adj_N', 'AD']:
    model = mod.CompositionalAnalysis(
        patient_adata_overlap,
        formula=f"C(tissue_type, Treatment('{ref_tissue}'))",
        reference_cell_type=ref_cell_type
    )
    result = model.sample_hmc(num_results=20000, num_burnin=5000)
    cred = result.credible_effects()
    print(f"\n--- Reference tissue: {ref_tissue} ---")
    print(cred)

    keep = pair_source[ref_tissue]
    for (covariate, cell_type), is_credible in cred.items():
        test_tissue = covariate.split('[T.')[1].rstrip(']')
        if test_tissue not in keep:
            continue
        key = f'{ref_tissue} vs {test_tissue}'
        if key not in sccoda_results:
            sccoda_results[key] = {}
        sccoda_results[key][cell_type] = is_credible

# Build df
credible_overlap = pd.DataFrame(sccoda_results, index=CATEGORY_ORDER)

# Reorder columns
col_order = [
    'Dist_N vs Adj_N', 'Dist_N vs AD', 'Dist_N vs CA',
    'Adj_N vs AD', 'Adj_N vs CA', 'AD vs CA'
]
credible_overlap = credible_overlap[[c for c in col_order if c in credible_overlap.columns]]

print("\nCombined credible effects:")
print(credible_overlap)

# --- Dot plot ---
fig, ax = plt.subplots(figsize=(10, 6))

for i, cat in enumerate(CATEGORY_ORDER):
    for j, comp in enumerate(credible_overlap.columns):
        if credible_overlap.loc[cat, comp]:
            ax.scatter(j, i, s=200, c=[CATEGORY_COLORS[cat]],
                       edgecolors='black', linewidth=1, zorder=3)
        else:
            ax.scatter(j, i, s=200, facecolors='none',
                       edgecolors='lightgray', linewidth=1, zorder=3)

ax.set_yticks(range(len(CATEGORY_ORDER)))
ax.set_yticklabels(CATEGORY_ORDER, fontsize=12, fontweight='bold')
ax.set_xticks(range(len(credible_overlap.columns)))
ax.set_xticklabels(credible_overlap.columns, rotation=45, ha='right', fontsize=12)
ax.set_xlim(-0.5, len(credible_overlap.columns) - 0.5)
ax.set_ylim(-0.5, len(CATEGORY_ORDER) - 0.5)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='both', alpha=0.2)

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markersize=12, markeredgecolor='black', label='Credible'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
               markersize=12, markeredgecolor='lightgray', label='Not credible'),
]
ax.legend(handles=legend_elements, loc='upper left',
          bbox_to_anchor=(1.02, 1), frameon=True, fontsize=11)

ax.set_xlabel('Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'overlap_credibility_dotplot_patient.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
def make_spatialdata_dict(spatialdata_dir):
    """Load all filtered/normalized SpatialData .zarr files into a dictionary."""
    spatialdata_dict = {}
    entries = [f for f in os.listdir(spatialdata_dir) if f.startswith("fil")]
    for filename in sorted(entries):
        sdata = sd.read_zarr(f"{spatialdata_dir}/{filename}")
        key = filename.replace('filtered_normalized_xenium_', '').replace('.zarr', '')
        spatialdata_dict[key] = sdata
    return spatialdata_dict


def transfer_obs_columns(spatialdata_dict, reference_adata, columns):
    """Transfer annotation columns from a reference AnnData to SpatialData tables (join on cell_id)."""
    ref_obs = reference_adata.obs.set_index('cell_id')[columns]
    for section_key, sdata in spatialdata_dict.items():
        table = sdata.tables['table']
        for col in columns:
            if col in ref_obs.columns:
                table.obs[col] = table.obs['cell_id'].map(ref_obs[col])
    return spatialdata_dict


spatialdata_dict = make_spatialdata_dict(SDATA_DIR)
print(f"Loaded {len(spatialdata_dict)} SpatialData sections")




# Identify valid (section, core) pairs
MIN_EPI_CELLS        = 100       # Minimum epithelial cells per core

epi_obs = all_cells_adata.obs[all_cells_adata.obs['annotation_final_fine_cd8'] == 'Epithelial']
epi_counts = epi_obs.groupby(['batch', 'core_id']).size()
valid_pairs = epi_counts[epi_counts >= MIN_EPI_CELLS].reset_index()
valid_section_cores = set(zip(valid_pairs['batch'], valid_pairs['core_id']))
print(f"Valid (section, core) pairs: {len(valid_section_cores)}")

# Filter spatialdata_dict — strict section assignment
valid_cell_ids = set(all_cells_adata.obs['cell_id'].unique())

filtered_spatialdata_dict = {}
for section_key, sdata in spatialdata_dict.items():
    table = sdata.tables['table']
    valid_cores_for_section = {
        core for (batch, core) in valid_section_cores if batch == section_key
    }
    mask = (
        table.obs['cell_id'].isin(valid_cell_ids)
        & table.obs['core_id'].isin(valid_cores_for_section)
    )
    if mask.sum() > 0:
        sdata.tables['table'] = table[mask].copy()
        filtered_spatialdata_dict[section_key] = sdata
        print(f"  {section_key}: {mask.sum()} cells, "
              f"{sdata.tables['table'].obs['core_id'].nunique()} cores")
    else:
        print(f"  {section_key}: dropped (no valid cells)")

# Verify no duplicate core assignments across sections
core_to_sections = {}
for section_key, sdata in filtered_spatialdata_dict.items():
    for core in sdata.tables['table'].obs['core_id'].unique():
        core_to_sections.setdefault(core, []).append(section_key)
duplicated = {c: s for c, s in core_to_sections.items() if len(s) > 1}
print(f"\nCores in multiple sections: {len(duplicated)}")
for core, sections in sorted(duplicated.items()):
    print(f"  {core}: {sections}")


TRANSFER_COLUMNS = [
    'annotation_final_fine', 'annotation_final_coarse',
    'tissue_type_cell_level', 'tissue_type_dysplasia_cell_level',
    'mixed_core_tissue_type', 'mixed_core_dysplasia',
    'tissue_type_cell_level_normal_split',
    'tissue_type_dysplasia_cell_level_normal_split',
    'epithelial_0.8',
    'annotation_final_fine_cd8',
    'GDF15_more_than_1_transcript', 'senepy_intestine_epi_0',
    'stemness_score', 'senepy_high', 'core_id', 'patient_id','GDF15_dense',
 'GDF15_morans_quad',
 'GDF15_morans_pval',
 'GDF15_morans_Ii',
 'stemness_score_dense',
 'stemness_score_morans_quad',
 'stemness_score_morans_pval',
 'stemness_score_morans_Ii',
 'senepy_intestine_epi_0_dense',
 'senepy_intestine_epi_0_morans_quad',
 'senepy_intestine_epi_0_morans_pval',
 'senepy_intestine_epi_0_morans_Ii'
]

filtered_spatialdata_dict = transfer_obs_columns(
    filtered_spatialdata_dict, all_cells_adata, TRANSFER_COLUMNS
)

# Sanity check
for section_key, sdata in filtered_spatialdata_dict.items():
    table = sdata.tables['table']
    n_epi = (table.obs['annotation_final_fine_cd8'] == 'Epithelial').sum()
    n_total = len(table)
    print(f"{section_key}: {n_total} total, {n_epi} epithelial ({100*n_epi/n_total:.1f}%)")

# Normal cores: stemness + SenePy + GDF15 transcripts

In [ ]:
sc.settings._vector_friendly = True

def plot_normal_cores_combined(
    filtered_spatialdata_dict,
    core_ids,
    tissue_col="tissue_type_cell_level_normal_split",
    feat_stem="stemness_score_dense",
    feat_sene="senepy_intestine_epi_0_dense",
    points_key="transcripts",
    gene_col="feature_name",
    transcript_gene="GDF15",
    color_stem="#44c3e3",
    color_sene="#F79F1F",
    color_other_epi="#d3d3d3",
    color_non_epi="#fafafa",
    color_transcript="#EA2027",
    dot_size_dense=12,
    dot_size_other=4,
    dot_size_non_epi=2,
    dot_size_transcript=6,
    alpha_dense=1.0,
    alpha_other_epi=0.6,
    alpha_non_epi=0.3,
    alpha_transcript=1.0,
    figsize_per_subplot=(4, 4),
    save_path=None,
):
    nrows = len(core_ids)
    ncols = 2
    fig, all_axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_subplot[0] * ncols, figsize_per_subplot[1] * nrows),
        squeeze=False,
    )

    def _clean_ax(ax):
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for row_idx, core_id in enumerate(core_ids):
        sdata_match = None
        core_obs = None
        section_key = None
        for sk, sdata in filtered_spatialdata_dict.items():
            table = sdata.tables['table']
            mask = table.obs['core_id'] == core_id
            if mask.any():
                sdata_match = sdata
                core_obs = table.obs[mask].copy()
                section_key = sk
                break

        if core_obs is None:
            print(f"Core {core_id} not found")
            for ax in all_axes[row_idx]:
                ax.set_visible(False)
            continue

        cell_shapes = sdata_match.shapes['cell_boundaries']
        core_spatial = cell_shapes[cell_shapes.index.isin(core_obs['cell_id'].values)]
        coord_map = {
            i: (g.centroid.x, g.centroid.y)
            for i, g in zip(core_spatial.index, core_spatial.geometry)
        }
        core_obs['cx'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan,))[0])
        core_obs['cy'] = core_obs['cell_id'].map(lambda c: coord_map.get(c, (np.nan, np.nan))[1])
        core_obs = core_obs.dropna(subset=['cx', 'cy'])

        tx_x, tx_y = np.array([]), np.array([])
        if points_key in sdata_match.points:
            import dask.dataframe as dd
            pts = sdata_match.points[points_key]
            if isinstance(pts, dd.DataFrame):
                pts = pts.compute()
            gdf15_pts = pts[pts[gene_col] == transcript_gene]
            xmin, ymin = core_obs['cx'].min(), core_obs['cy'].min()
            xmax, ymax = core_obs['cx'].max(), core_obs['cy'].max()
            pad = 50
            gdf15_pts = gdf15_pts[
                (gdf15_pts['x'] >= xmin - pad) & (gdf15_pts['x'] <= xmax + pad) &
                (gdf15_pts['y'] >= ymin - pad) & (gdf15_pts['y'] <= ymax + pad)
            ]
            tx_x = gdf15_pts['x'].values
            tx_y = gdf15_pts['y'].values
            print(f"  Core {core_id}: {len(tx_x)} GDF15 transcripts")
        else:
            print(f"  Warning: '{points_key}' not found for {section_key}")

        is_epi = core_obs['annotation_final_fine'] == 'Epithelial'
        is_non_epi = ~is_epi
        has_stem = (core_obs[feat_stem] == True) if feat_stem in core_obs.columns else pd.Series(False, index=core_obs.index)
        has_sene = (core_obs[feat_sene] == True) if feat_sene in core_obs.columns else pd.Series(False, index=core_obs.index)

        def _draw_bg(ax):
            if is_non_epi.any():
                ax.scatter(
                    core_obs.loc[is_non_epi, 'cx'], core_obs.loc[is_non_epi, 'cy'],
                    s=dot_size_non_epi, c=color_non_epi, alpha=alpha_non_epi,
                    linewidths=0, zorder=2, rasterized=True,
                )

        def _draw_transcripts(ax):
            if len(tx_x) > 0:
                ax.scatter(
                    tx_x, tx_y,
                    s=dot_size_transcript, c=color_transcript, alpha=alpha_transcript,
                    marker='x', linewidths=0.5, zorder=10,
                )

        axes = all_axes[row_idx]

        # Panel 0: Stemness
        ax = axes[0]
        _draw_bg(ax)
        is_dense = is_epi & has_stem
        is_other = is_epi & ~is_dense
        if is_other.any():
            ax.scatter(core_obs.loc[is_other, 'cx'], core_obs.loc[is_other, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if is_dense.any():
            ax.scatter(core_obs.loc[is_dense, 'cx'], core_obs.loc[is_dense, 'cy'],
                       s=dot_size_dense, c=color_stem, alpha=alpha_dense,
                       linewidths=0, zorder=5)
        _draw_transcripts(ax)
        if row_idx == 0:
            ax.set_title("Stemness", fontsize=11, fontweight='bold', color=color_stem)
        _clean_ax(ax)

        # Panel 1: SenePy
        ax = axes[1]
        _draw_bg(ax)
        is_dense = is_epi & has_sene
        is_other = is_epi & ~is_dense
        if is_other.any():
            ax.scatter(core_obs.loc[is_other, 'cx'], core_obs.loc[is_other, 'cy'],
                       s=dot_size_other, c=color_other_epi, alpha=alpha_other_epi,
                       linewidths=0, zorder=3, rasterized=True)
        if is_dense.any():
            ax.scatter(core_obs.loc[is_dense, 'cx'], core_obs.loc[is_dense, 'cy'],
                       s=dot_size_dense, c=color_sene, alpha=alpha_dense,
                       linewidths=0, zorder=5)
        _draw_transcripts(ax)
        if row_idx == 0:
            ax.set_title("SenePy", fontsize=11, fontweight='bold', color=color_sene)
        _clean_ax(ax)

        # Row label
        tissue = core_obs.get(tissue_col, pd.Series(dtype=str)).dropna().unique()
        tissue_str = tissue[0] if len(tissue) > 0 else "?"
        axes[0].set_ylabel(f"{core_id}\n[{tissue_str}]", fontsize=10, fontweight='bold', rotation=0, labelpad=60, va='center')

    fig.tight_layout()

    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"Saved to {save_path}")

    plt.show()


plot_normal_cores_combined(
    filtered_spatialdata_dict,
    core_ids=["N1-N-2", "L2-N-1", "P1-N-2"],
    save_path=str(P.results.figures / "figure_3" / "normal_cores_stem_sene_gdf15.pdf"),
)

sc.settings._vector_friendly = False

# Overlap category analysis

In [ ]:
epi_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_20_50_harmony_batch_05_pt_05_ssg.h5ad"))

In [ ]:
# ── Compute overlap categories from all_cells_adata ──────────────────────────
# The _dense columns are already in all_cells_adata.obs from the Moran's analysis.
# We classify each epithelial cell into one of 8 overlap categories.

DENSE_FEATURES = ["GDF15_dense", "stemness_score_dense", "senepy_intestine_epi_0_dense"]
FEATURE_SHORT = {
    "GDF15_dense": "GDF15",
    "stemness_score_dense": "Stem",
    "senepy_intestine_epi_0_dense": "SenePy",
}

# Work on epithelial cells only
epi_mask = all_cells_adata.obs["annotation_final_fine_cd8"] == "Epithelial"
epi_obs = all_cells_adata.obs.loc[epi_mask].copy()

# Build boolean columns (handle NaN safely)
for feat in DENSE_FEATURES:
    epi_obs[feat] = epi_obs[feat].fillna(False).astype(bool)

has_gdf15  = epi_obs["GDF15_dense"]
has_stem   = epi_obs["stemness_score_dense"]
has_senepy = epi_obs["senepy_intestine_epi_0_dense"]

# Assign overlap category
conditions = []
labels = []

# Triple
conditions.append(has_gdf15 & has_stem & has_senepy)
labels.append("GDF15+Stem+SenePy")

# Pairwise only
conditions.append(has_gdf15 & has_stem & ~has_senepy)
labels.append("GDF15+Stem")

conditions.append(has_gdf15 & ~has_stem & has_senepy)
labels.append("GDF15+SenePy")

conditions.append(~has_gdf15 & has_stem & has_senepy)
labels.append("Stem+SenePy")

# Single only
conditions.append(has_gdf15 & ~has_stem & ~has_senepy)
labels.append("GDF15_only")

conditions.append(~has_gdf15 & has_stem & ~has_senepy)
labels.append("Stem_only")

conditions.append(~has_gdf15 & ~has_stem & has_senepy)
labels.append("SenePy_only")

# None
conditions.append(~has_gdf15 & ~has_stem & ~has_senepy)
labels.append("None")

overlap_cat = pd.Series("None", index=epi_obs.index, dtype="object")
for cond, label in zip(conditions, labels):
    overlap_cat.loc[cond] = label

epi_obs["overlap_category"] = pd.Categorical(overlap_cat, categories=labels, ordered=False)

# Summarize
print("Overlap category counts (epithelial cells in all_cells_adata):")
print(epi_obs["overlap_category"].value_counts().to_string())
print(f"\nTotal epithelial cells classified: {len(epi_obs)}")

In [ ]:
# ── Transfer overlap_category to epi_adata ───────────────────────────────────
# Join on cell_id

overlap_map = epi_obs.set_index("cell_id")["overlap_category"]

epi_adata.obs["overlap_category"] = (
    epi_adata.obs["cell_id"]
    .map(overlap_map)
    .astype("category")
)

n_mapped = epi_adata.obs["overlap_category"].notna().sum()
n_total = len(epi_adata)
print(f"Mapped {n_mapped}/{n_total} cells in epi_adata ({100*n_mapped/n_total:.1f}%)")
print(epi_adata.obs["overlap_category"].value_counts())

In [ ]:
# ── DEG comparison 1: Stem+SenePy vs (Stem_only + SenePy_only) — by tissue type ──

tissue_col = "tissue_type_cell_level_normal_split"
tissue_types = ["Adj_N", "AD", "CA"]

deg_results = {}

for tissue in tissue_types:
    print(f"\n{'='*80}")
    print(f"  Tissue type: {tissue}")
    print(f"{'='*80}\n")

    # Subset 
    mask = (
        epi_adata.obs[tissue_col].eq(tissue)
        & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
    )
    adata_c1 = epi_adata[mask].copy()

    adata_c1.obs["deg_group"] = (
        adata_c1.obs["overlap_category"]
        .map(lambda x: "Stem+SenePy" if x == "Stem+SenePy" else "Stem_or_SenePy_only")
    )
    print(f"Comparison 1 ({tissue}): {adata_c1.obs['deg_group'].value_counts().to_string()}")

    # Check both groups have cells
    if adata_c1.obs["deg_group"].nunique() < 2:
        print(f"  ⚠ Skipping {tissue} — fewer than 2 groups present")
        continue

    sc.tl.rank_genes_groups(adata_c1, groupby="deg_group", method="wilcoxon",
                            reference="Stem_or_SenePy_only", pts=True)

    deg_c1 = sc.get.rank_genes_groups_df(adata_c1, group="Stem+SenePy")
    deg_c1["comparison"] = "vs_Stem_or_SenePy_only"
    deg_c1["tissue"] = tissue
    deg_results[tissue] = deg_c1

    
    top20 = deg_c1.head(20)[["names", "logfoldchanges", "pvals_adj"]].copy()
    top20.columns = ["gene", "log2FC", "padj"]
    print(f"\nTop 20 DEGs ({tissue}):")
    print(top20.to_string(index=False))

    
    top20_genes_c1 = deg_c1.head(20)["names"].tolist()

    adata_c1_3way = epi_adata[mask].copy()
    adata_c1_3way.obs["deg_group"] = adata_c1_3way.obs["overlap_category"].astype(str).astype(object)


deg_all_tissues = pd.concat(deg_results.values(), ignore_index=True)

# DEG dotplots by tissue

In [ ]:
import os

save_dir = str(P.results.figures / "figure_3")
os.makedirs(save_dir, exist_ok=True)

tissue_col = "tissue_type_cell_level_normal_split"
tissue_types = ["Adj_N", "AD", "CA"]

for tissue in tissue_types:
    mask = (
        epi_adata.obs[tissue_col].eq(tissue)
        & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
    )

    top20_genes_c1 = deg_results[tissue].head(5)["names"].tolist()

    adata_c1_3way = epi_adata[mask].copy()
    adata_c1_3way.obs["deg_group"] = pd.Categorical(
        adata_c1_3way.obs["overlap_category"].astype(str),
        categories=["SenePy_only", "Stem_only", "Stem+SenePy"],
        ordered=True,
    )

    dp = sc.pl.dotplot(
        adata_c1_3way,
        var_names=top20_genes_c1,
        groupby="deg_group",
        title=f"{tissue}: Stem+SenePy vs Stem_only/SenePy_only — Top 5 DEGs",
        show=False,
        return_fig=True,
    )
    dp.savefig(os.path.join(save_dir, f"dotplot_top5_DEGs_{tissue}.pdf"),
               dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

# Violin plots

In [ ]:
import matplotlib.pyplot as plt
import os

save_dir = str(P.results.figures / "supp_12")
os.makedirs(save_dir, exist_ok=True)

CATEGORY_COLORS = {
    "GDF15_only":         "#EA2027",
    "Stemness_only":      "#44c3e3",
    "SenePy_only":        "#F79F1F",
    'GDF15+Stem':   "#5758BB",
    'Stem+SenePy':  "#046e27",
    'GDF15+SenePy':     "#EE5A24",
    'GDF15+Stem+SenePy':          "#000000",
    "None":          "#d3d3d3",
}


def plot_violin_all_groups_by_tissue(epi_adata, gene, save_dir):
    tissue_col = "tissue_type_cell_level_normal_split"
    tissue_types = ["AD", "CA"]

    # Build group column
    epi_adata.obs["overlap_group"] = epi_adata.obs["overlap_category"].astype(str)
    known_groups = epi_adata.obs["overlap_category"].dropna().unique()
    known_groups = [g for g in known_groups if g not in ("nan", "None", "")]
    other_mask = ~epi_adata.obs["overlap_group"].isin(known_groups)
    epi_adata.obs.loc[other_mask, "overlap_group"] = "Other Epi"

   
    ordered_known = [g for g in CATEGORY_COLORS if g in known_groups]
    extras = [g for g in known_groups if g not in CATEGORY_COLORS]
    group_order = ordered_known + sorted(extras) + ["Other Epi"]

    # Build color list
    cmap = plt.cm.get_cmap("tab10")
    auto_idx = 0
    group_colors = []
    for g in group_order:
        if g in CATEGORY_COLORS:
            group_colors.append(CATEGORY_COLORS[g])
        else:
            group_colors.append(cmap(auto_idx % 10))
            auto_idx += 1

    
    fig, axes = plt.subplots(1, 2, figsize=(6 * len(tissue_types), 6))

    for ax, tissue in zip(axes, tissue_types):
        mask = epi_adata.obs[tissue_col].eq(tissue)
        adata_sub = epi_adata[mask].copy()
        adata_sub.obs["deg_group"] = pd.Categorical(
            adata_sub.obs["overlap_group"].astype(str),
            categories=group_order,
            ordered=True,
        )
        
        adata_sub.uns["deg_group_colors"] = group_colors

        sc.pl.violin(
            adata_sub,
            keys=gene,
            groupby="deg_group",
            order=group_order,
            rotation=45,
            stripplot=False,
            inner=None,
            ax=ax,
            show=False,
        )

        ax.set_title(f"{tissue}: {gene}", fontsize=12)
        ax.set_xlabel("")
        ax.set_xticks(range(len(group_order)))
        ax.set_xticklabels(group_order, rotation=45, ha='right', fontsize=8)

    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f"violin_{gene}_TA_CA.pdf"),
                dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


plot_violin_all_groups_by_tissue(epi_adata, "OLFM4", save_dir)
plot_violin_all_groups_by_tissue(epi_adata, "APCDD1", save_dir)

In [ ]:
import matplotlib.pyplot as plt
import os

save_dir = str(P.results.figures / "figure_3")
os.makedirs(save_dir, exist_ok=True)

def plot_violin_by_tissue_save(epi_adata, gene, save_dir):
    tissue_col = "tissue_type_cell_level_normal_split"
    tissue_types = ["Adj_N", "AD", "CA"]
    group_order = ["Stem_only", "SenePy_only", "Stem+SenePy"]
    group_colors = ["#44c3e3", "#F79F1F", "#046e27"]

    # Combined plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, tissue in zip(axes, tissue_types):
        mask = (
            epi_adata.obs[tissue_col].eq(tissue)
            & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
        )

        adata_sub = epi_adata[mask].copy()
        adata_sub.obs["deg_group"] = pd.Categorical(
            adata_sub.obs["overlap_category"].astype(str),
            categories=group_order,
            ordered=True,
        )

        sc.pl.violin(
            adata_sub,
            keys=gene,
            groupby="deg_group",
            order=group_order,
            palette=group_colors,
            rotation=45,
            stripplot=False,
            inner=None,
            ax=ax,
            show=False,
        )

        ax.set_title(f"{tissue}: {gene}", fontsize=12)
        ax.set_xlabel("")

    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f"violin_{gene}_combined.pdf"),
                dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    # Individual plots
    for tissue in tissue_types:
        mask = (
            epi_adata.obs[tissue_col].eq(tissue)
            & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
        )

        adata_sub = epi_adata[mask].copy()
        adata_sub.obs["deg_group"] = pd.Categorical(
            adata_sub.obs["overlap_category"].astype(str),
            categories=group_order,
            ordered=True,
        )

        fig, ax = plt.subplots(figsize=(5, 4))

        sc.pl.violin(
            adata_sub,
            keys=gene,
            groupby="deg_group",
            order=group_order,
            palette=group_colors,
            rotation=45,
            stripplot=False,
            inner=None,
            ax=ax,
            show=False,
        )

        ax.set_title(f"{tissue}: {gene}", fontsize=12)
        ax.set_xlabel("")
        plt.tight_layout()
        fig.savefig(os.path.join(save_dir, f"violin_{gene}_{tissue}.pdf"),
                    dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)


plot_violin_by_tissue_save(epi_adata, "OLFM4", save_dir)
plot_violin_by_tissue_save(epi_adata, "APCDD1", save_dir)

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 1, figsize=(3, 5))

group_order = ["Stem_only", "SenePy_only", "Stem+SenePy"]
group_colors = ["#44c3e3", "#F79F1F", "#046e27"]
tissue_col = "tissue_type_cell_level_normal_split"

mask = (
    epi_adata.obs[tissue_col].eq("AD")
    & epi_adata.obs["overlap_category"].isin(["Stem+SenePy", "Stem_only", "SenePy_only"])
)
adata_sub = epi_adata[mask].copy()
adata_sub.obs["deg_group"] = pd.Categorical(
    adata_sub.obs["overlap_category"].astype(str),
    categories=group_order,
    ordered=True,
)

for ax, gene in zip(axes, ["OLFM4", "APCDD1"]):
    gene_expr = adata_sub[:, gene].X.toarray().flatten() if hasattr(adata_sub[:, gene].X, "toarray") else np.asarray(adata_sub[:, gene].X).flatten()
    plot_df = pd.DataFrame({"expression": gene_expr, "group": adata_sub.obs["deg_group"].values})

    sns.violinplot(
        data=plot_df,
        y="group",
        x="expression",
        order=group_order,
        palette=group_colors,
        orient="h",
        inner=None,
        cut=0,
        ax=ax,
    )
    ax.set_title(f"AD: {gene}", fontsize=12)
    ax.set_ylabel("")
    ax.set_xlabel(gene)

plt.tight_layout()
fig.savefig(os.path.join(save_dir, "violin_TA_OLFM4_APCDD1_horizontal.pdf"),
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()